In [ ]:
# -*- coding: utf-8 -*-
"""
통합 ESG-GNN 파이프라인: 실제 BEA 데이터 + 고급 GNN 아키텍처
- 기존 전처리 파이프라인 + 고급 GNN 모델 통합
- 실제 산업연관표와 BEA 6개 지표 활용
- 차별화된 아키텍처 구현
"""

import os
import re
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from typing import Dict, List, Tuple, Optional, Union
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv, SAGEConv, GCNConv
from torch_geometric.data import Data

# ---------------------------------------------------------------------
# 기존 전처리 함수들 (수정된 버전)
# ---------------------------------------------------------------------

# 파일 경로 설정
IOT_PATH = os.getenv("ICIO_USA_IO_PATH", "data/external/NATIOTTL/USA2020ttl.csv")
FILES6 = {
    "RVA": os.getenv("BEA_REAL_VALUE_ADDED_PATH", "data/external/bea/Real Value Added by Industry.xlsx"),
    "RII": os.getenv("BEA_REAL_INTERMEDIATE_INPUT_PATH", "data/external/bea/Real Intermediate Input by Industry.xlsx"), 
    "PII": os.getenv("BEA_INTERMEDIATE_INPUT_PRICE_INDEX_PATH", "data/external/bea/Chain-Type Price Indexes for Intermediate Inputs by Industry.xlsx"),
    "RGO": os.getenv("BEA_REAL_GROSS_OUTPUT_PATH", "data/external/bea/Real Gross Output by Industry.xlsx"),
    "PGO": os.getenv("BEA_GROSS_OUTPUT_PRICE_INDEX_PATH", "data/external/bea/Chain-Type Price Indexes for Gross Output by Industry.xlsx"),
    "GO" : os.getenv("BEA_GROSS_OUTPUT_PATH", "data/external/bea/Gross Output by Industry.xlsx"),
}
DEFAULT_HEADER_ROW = 4

def load_edge_list_from_iot(iot_path: str) -> pd.DataFrame:
    """산업연관표 → edge list 변환"""
    df = pd.read_csv(iot_path)
    df = df.rename(columns={df.columns[0]: "source"})
    edge = df.melt(id_vars="source", var_name="target", value_name="weight")
    edge = edge.dropna(subset=["weight"])
    edge = edge[edge["weight"] > 0]

    edge["source"] = (edge["source"]
                      .str.replace(r"^TTL_", "", regex=True)
                      .str.strip()
                      .str.upper())
    edge["target"] = (edge["target"]
                      .str.replace(r"^TTL_", "", regex=True)
                      .str.strip()
                      .str.upper())
    return edge.reset_index(drop=True)

def _read_any_table(file_path: str, header: int) -> pd.DataFrame:
    """파일 확장자별 읽기"""
    ext = os.path.splitext(file_path)[1].lower()
    if ext in [".xlsx", ".xls"]:
        return pd.read_excel(file_path, header=header)
    return pd.read_csv(file_path, header=header)

def load_and_clean_bea(file_path: str, prefix: str, header: int = DEFAULT_HEADER_ROW) -> pd.DataFrame:
    """BEA 파일 로드 및 정리 (개선된 버전)"""
    df = _read_any_table(file_path, header)
    
    # 산업명 컬럼 찾기
    industry_col = None
    for i, col in enumerate(df.columns[:3]):
        if df[col].dtype == 'object' and df[col].notna().sum() > 10:
            industry_col = col
            break
    
    if industry_col is None:
        industry_col = df.columns[1]
    
    df.rename(columns={industry_col: "Industry"}, inplace=True)
    df["Industry"] = df["Industry"].astype(str).str.strip()
    df = df.dropna(subset=["Industry"])
    df = df[df["Industry"] != ""]
    df = df[~df["Industry"].str.contains("Addenda|Legend|Footnotes|Consists of|technology-producing|Note:|Source:", na=False, case=False)]
    
    if "Line" in df.columns:
        df = df[df["Line"].notna()]

    # 시계열 컬럼 찾기 (분기, 연도, 혼합 형태)
    time_cols = []
    for col in df.columns:
        if isinstance(col, str):
            if re.search(r'\d{4}.*Q[1-4]', col, re.IGNORECASE):
                time_cols.append(col)
            elif re.search(r'^\d{4}$', str(col).strip()):
                time_cols.append(col)
            elif col.replace('.', '').replace('-', '').isdigit() and len(col) >= 4:
                time_cols.append(col)
    
    # 숫자 컬럼 추가 검색
    for col in df.columns:
        if col not in time_cols and col != "Industry":
            try:
                numeric_data = pd.to_numeric(df[col], errors='coerce')
                if numeric_data.notna().sum() / len(df) > 0.5:
                    time_cols.append(col)
            except:
                continue
    
    if not time_cols:
        print(f"⚠️ {prefix}: 시계열 컬럼 미발견. 모든 숫자 컬럼 사용.")
        time_cols = [c for c in df.columns if c != "Industry" and c != "Line"]
    
    time_cols = sorted(time_cols)
    rename_dict = {c: f"{prefix}_{c}" for c in time_cols}
    df = df[["Industry"] + time_cols].rename(columns=rename_dict)

    # 수치화
    for c in df.columns:
        if c.startswith(prefix + "_"):
            df[c] = pd.to_numeric(df[c], errors="coerce")
    
    print(f"✅ {prefix}: {len(time_cols)}개 시계열 컬럼")
    return df

def _latest_col(df: pd.DataFrame, prefix: str) -> str:
    """최신 컬럼 선택"""
    cols = [c for c in df.columns if c.startswith(prefix + "_")]
    if not cols:
        raise ValueError(f"{prefix}: 시계열 컬럼 미발견. 컬럼: {list(df.columns)}")
    
    def extract_year_quarter(col_name):
        time_part = col_name.replace(prefix + "_", "")
        year_match = re.search(r'(\d{4})', time_part)
        year = int(year_match.group(1)) if year_match else 0
        quarter_match = re.search(r'Q([1-4])', time_part, re.IGNORECASE)
        quarter = int(quarter_match.group(1)) if quarter_match else 0
        return (year, quarter)
    
    cols_sorted = sorted(cols, key=extract_year_quarter, reverse=True)
    return cols_sorted[0]

# 확장된 산업명 매핑
NAME2CODE = {
    # 농림어업
    "Agriculture, forestry, fishing, and hunting": "A01_02",
    "Farms": "A01_02",
    "Crop and animal production": "A01_02",
    "Forestry, fishing, and related activities": "A03",
    "Forestry and logging": "A03",
    "Fishing, hunting, and trapping": "A03",
    "Support activities for agriculture and forestry": "A03",

    # 광업
    "Mining": "B07_08",
    "Mining, quarrying, and oil and gas extraction": "B05_06",
    "Oil and gas extraction": "B05_06",
    "Mining, except oil and gas": "B07_08",
    "Coal mining": "B07_08",
    "Metal ore mining": "B07_08",
    "Nonmetallic mineral mining and quarrying": "B07_08",
    "Support activities for mining": "B09",

    # 유틸리티/건설
    "Utilities": "D",
    "Electric power generation, transmission, and distribution": "D",
    "Natural gas distribution": "E",
    "Water, sewage and other systems": "E",
    "Construction": "F",
    "Construction of buildings": "F",
    "Heavy and civil engineering construction": "F",
    "Specialty trade contractors": "F",

    # 제조업
    "Manufacturing": "C10T12",
    "Durable goods": "C16",
    "Nondurable goods": "C10T12",
    "Food and beverage and tobacco products": "C10T12",
    "Food manufacturing": "C10T12",
    "Beverage and tobacco product manufacturing": "C10T12",
    "Textile mills and textile product mills": "C13T15",
    "Apparel and leather and allied products": "C13T15",
    "Wood products": "C16",
    "Wood product manufacturing": "C16",
    "Paper products": "C17_18", 
    "Paper manufacturing": "C17_18",
    "Printing and related support activities": "C17_18",
    "Petroleum and coal products": "C19",
    "Petroleum and coal product manufacturing": "C19",
    "Chemical products": "C20",
    "Chemical manufacturing": "C20",
    "Plastics and rubber products": "C22",
    "Nonmetallic mineral products": "C23",
    "Nonmetallic mineral product manufacturing": "C23",
    "Primary metals": "C24",
    "Primary metal manufacturing": "C24",
    "Fabricated metal products": "C25",
    "Fabricated metal product manufacturing": "C25",
    "Computer and electronic products": "C26",
    "Computer and electronic product manufacturing": "C26",
    "Electrical equipment, appliances, and components": "C27",
    "Electrical equipment, appliance, and component manufacturing": "C27",
    "Machinery": "C28",
    "Machinery manufacturing": "C28",
    "Motor vehicles, bodies and trailers, and parts": "C29",
    "Motor vehicle manufacturing": "C29",
    "Other transportation equipment": "C30",
    "Transportation equipment manufacturing": "C30",
    "Aerospace product and parts manufacturing": "C30",
    "Furniture and related products": "C31T33",
    "Furniture and related product manufacturing": "C31T33",
    "Miscellaneous manufacturing": "C31T33",

    # 도소매/운수
    "Wholesale trade": "G",
    "Retail trade": "G",
    "Trade": "G",
    "Transportation and warehousing": "H49",
    "Air transportation": "H51",
    "Rail transportation": "H49",
    "Water transportation": "H50",
    "Truck transportation": "H49",
    "Transit and ground passenger transportation": "H49",
    "Pipeline transportation": "H52",
    "Scenic and sightseeing transportation": "H49",
    "Support activities for transportation": "H52",
    "Postal service": "H53",
    "Couriers and messengers": "H53",
    "Warehousing and storage": "H53",

    # 정보통신/금융
    "Information": "J61",
    "Publishing industries (includes software)": "J58T60",
    "Publishing industries, except internet (includes software)": "J58T60",
    "Motion picture and sound recording industries": "J59T60",
    "Broadcasting and telecommunications": "J61",
    "Broadcasting (except internet)": "J61",
    "Telecommunications": "J61",
    "Data processing, internet publishing, and other information services": "J62_63",
    "Data processing, hosting, and related services": "J62_63",
    "Other information services": "J62_63",
    "Finance and insurance": "K",
    "Finance": "K64",
    "Insurance": "K65",
    "Funds, trusts, and other financial vehicles": "K66",
    "Real estate and rental and leasing": "L",
    "Real estate": "L68",
    "Rental and leasing services and lessors of intangible assets": "L",

    # 서비스업
    "Professional, scientific, and technical services": "M",
    "Professional and business services": "M",
    "Legal services": "M69",
    "Computer systems design and related services": "M",
    "Miscellaneous professional, scientific, and technical services": "M",
    "Management of companies and enterprises": "M",
    "Administrative and support services": "N",
    "Administrative and waste management services": "N",
    "Waste management and remediation services": "N",
    "Government": "O",
    "Federal": "O",
    "State and local": "O",
    "Educational services": "P",
    "Health care and social assistance": "Q",
    "Health care": "Q",
    "Social assistance": "Q",
    "Ambulatory health care services": "Q",
    "Hospitals": "Q",
    "Nursing and residential care facilities": "Q",
    "Arts, entertainment, and recreation": "R",
    "Arts, entertainment, recreation, accommodation, and food services": "I",
    "Performing arts, spectator sports, and related industries": "R",
    "Museums, historical sites, and similar institutions": "R",
    "Amusement, gambling, and recreation industries": "R",
    "Accommodation and food services": "I",
    "Accommodation": "I",
    "Food services and drinking places": "I",
    "Other services, except government": "S",
    "Other services (except government)": "S",
    "Repair and maintenance": "S",
    "Personal and laundry services": "S",
    "Religious, grantmaking, civic, professional, and similar organizations": "S",
    "Private households": "T",
    "Households": "T",
}

def build_node_features_from_6(files6: dict, name2code: dict,
                               header: int = DEFAULT_HEADER_ROW,
                               scale: bool = True, 
                               lag_periods: int = 4) -> tuple[pd.DataFrame, list[str], dict]:
    """
    6개 BEA 파일을 병합하여 노드 특성 생성 (시차 특성 포함)
    반환: (node_df, feat_cols, temporal_data)
    """
    
    dfs_latest = []
    dfs_temporal = {}  # 시차 데이터 저장
    feat_order = ["RVA", "RII", "PII", "RGO", "PGO", "GO"]
    
    for prefix in feat_order:
        path = files6[prefix]
        print(f"\n처리 중: {prefix} - {os.path.basename(path)}")
        
        try:
            df = load_and_clean_bea(path, prefix, header=header)
            
            # 최신 데이터 (단일 컬럼)
            lastc = _latest_col(df, prefix)
            df_latest = df[["Industry", lastc]].rename(columns={lastc: prefix})
            dfs_latest.append(df_latest)
            
            # 시차 데이터 (여러 컬럼)
            time_cols = [c for c in df.columns if c.startswith(prefix + "_")]
            if len(time_cols) >= lag_periods:
                # 최신 lag_periods 개 컬럼 선택
                recent_cols = sorted(time_cols)[-lag_periods:]
                df_temporal = df[["Industry"] + recent_cols]
                dfs_temporal[prefix] = df_temporal
            
            print(f"✅ {prefix}: 최신 '{lastc}' + 시차 {len(time_cols)} 컬럼")
            
        except Exception as e:
            print(f"❌ {prefix} 처리 실패: {e}")
            empty_df = pd.DataFrame({"Industry": [], prefix: []})
            dfs_latest.append(empty_df)

    # 최신 데이터 병합
    node_raw = dfs_latest[0]
    for i, df in enumerate(dfs_latest[1:], 1):
        if len(df) > 0:
            node_raw = node_raw.merge(df, on="Industry", how="outer")
        else:
            node_raw[feat_order[i]] = np.nan

    # 산업명 → 코드 매핑
    node_raw["code"] = node_raw["Industry"].map(name2code)
    
    # 매핑 누락 확인
    unmapped = node_raw[node_raw["code"].isna()]["Industry"].unique()
    if len(unmapped) > 0:
        print(f"\n⚠️ 매핑 누락 산업 {len(unmapped)}개:")
        for industry in unmapped[:10]:
            print(f"  - '{industry}'")
        if len(unmapped) > 10:
            print(f"  ... 그리고 {len(unmapped)-10}개 더")
    
    node = (node_raw
            .dropna(subset=["code"])
            .drop(columns=["Industry"])
            .groupby("code", as_index=False)
            .mean())

    # 시차 데이터 처리
    temporal_data = {}
    for prefix, df_temp in dfs_temporal.items():
        if len(df_temp) > 0:
            df_temp["code"] = df_temp["Industry"].map(name2code)
            df_temp_clean = (df_temp
                           .dropna(subset=["code"])
                           .drop(columns=["Industry"])
                           .groupby("code", as_index=False)
                           .mean())
            temporal_data[prefix] = df_temp_clean

    feat_cols = feat_order[:]
    if scale:
        scaler = StandardScaler()
        node[feat_cols] = node[feat_cols].fillna(0.0)
        node[feat_cols] = scaler.fit_transform(node[feat_cols].astype(float))

    return node, feat_cols, temporal_data


# ---------------------------------------------------------------------
# 고급 ESG 데이터 프로세서 (통합 버전)
# ---------------------------------------------------------------------

class IntegratedESGProcessor:
    """실제 BEA 데이터를 활용한 고급 ESG 전처리기"""
    
    def __init__(self, iot_path: str, files6: dict, name2code: dict, lag_periods: int = 4):
        self.iot_path = iot_path
        self.files6 = files6
        self.name2code = name2code
        self.lag_periods = lag_periods
        self.scalers = {}
        
    def process_full_pipeline(self) -> Dict[str, any]:
        """전체 데이터 처리 파이프라인"""
        print("🔄 통합 ESG 데이터 처리 시작...")
        
        # 1. 엣지 리스트 생성
        edge_df = load_edge_list_from_iot(self.iot_path)
        print(f"✅ Edge list: {edge_df.shape}")
        
        # 2. 노드 특성 생성 (최신 + 시차)
        node_df, feat_cols, temporal_data = build_node_features_from_6(
            self.files6, self.name2code, scale=True, lag_periods=self.lag_periods
        )
        print(f"✅ Node features: {node_df.shape}")
        
        # 3. 이중 방향 엣지 생성
        edge_index_up, edge_index_down, edge_weight_up, edge_weight_down = \
            self.create_bidirectional_edges(edge_df)
        
        # 4. 탄성 가중치 계산
        elasticity_weights = self.compute_elasticity_weights(node_df, edge_df, feat_cols)
        
        # 5. ESG 채널 생성
        esg_channels = self.create_esg_channels(node_df, feat_cols)
        
        # 6. Leontief prior 계산
        leontief_prior = self.compute_leontief_prior(edge_df, node_df)
        
        # 7. 시차 특성 텐서 생성
        x_temporal = self.create_temporal_tensor(temporal_data, node_df, feat_cols)
        
        # 8. 최종 노드 특성 텐서
        x = torch.tensor(node_df[feat_cols].values, dtype=torch.float)
        
        # 9. 코드 매핑 정보
        codes = node_df["code"].tolist()
        code2id = {c: i for i, c in enumerate(codes)}
        
        return {
            'x': x,
            'edge_index_up': edge_index_up,
            'edge_index_down': edge_index_down,
            'edge_weight_up': edge_weight_up,
            'edge_weight_down': edge_weight_down,
            'elasticity_weights': elasticity_weights,
            'esg_channels': esg_channels,
            'leontief_prior': leontief_prior,
            'x_temporal': x_temporal,
            'codes': codes,
            'code2id': code2id,
            'feat_cols': feat_cols,
            'node_df': node_df,
            'edge_df': edge_df,
            'temporal_data': temporal_data
        }
    
    def create_bidirectional_edges(self, edge_df: pd.DataFrame) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
        """이중 방향 엣지 생성"""
        codes = sorted(set(edge_df["source"]) | set(edge_df["target"]))
        code2id = {c: i for i, c in enumerate(codes)}
        
        # 존재하는 코드만 필터링
        valid_edges = edge_df[
            edge_df["source"].isin(code2id.keys()) & 
            edge_df["target"].isin(code2id.keys())
        ]
        
        if len(valid_edges) == 0:
            print("⚠️ 유효한 엣지가 없습니다!")
            n = len(codes)
            return (torch.zeros((2, 0), dtype=torch.long),
                   torch.zeros((2, 0), dtype=torch.long),
                   torch.zeros(0, dtype=torch.float),
                   torch.zeros(0, dtype=torch.float))
        
        # Upstream: 공급자 → 수요자
        edge_index_up = torch.tensor([
            valid_edges["source"].map(code2id).values,
            valid_edges["target"].map(code2id).values
        ], dtype=torch.long)
        
        # Downstream: 수요자 → 공급자
        edge_index_down = torch.tensor([
            valid_edges["target"].map(code2id).values,
            valid_edges["source"].map(code2id).values
        ], dtype=torch.long)
        
        edge_weight_up = torch.tensor(valid_edges["weight"].values, dtype=torch.float)
        edge_weight_down = edge_weight_up.clone()
        
        print(f"✅ 이중 방향 엣지: Up {edge_index_up.shape[1]}, Down {edge_index_down.shape[1]}")
        return edge_index_up, edge_index_down, edge_weight_up, edge_weight_down
    
    def compute_elasticity_weights(self, node_df: pd.DataFrame, edge_df: pd.DataFrame, 
                                 feat_cols: List[str]) -> torch.Tensor:
        """탄성 가중치 계산"""
        # RII/RGO 비율로 투입 집약도 계산
        if "RII" in feat_cols and "RGO" in feat_cols:
            rii_idx = feat_cols.index("RII")
            rgo_idx = feat_cols.index("RGO")
            
            node_features = node_df[feat_cols].values
            input_intensity = node_features[:, rii_idx] / (node_features[:, rgo_idx] + 1e-8)
            
            # 가격 민감도 (PII 변화율)
            if "PII" in feat_cols:
                pii_idx = feat_cols.index("PII")
                price_sensitivity = np.abs(node_features[:, pii_idx])
                elasticity_scores = input_intensity * price_sensitivity
            else:
                elasticity_scores = input_intensity
        else:
            # 기본값: 균등 가중치
            elasticity_scores = np.ones(len(node_df))
        
        # 엣지별 탄성 할당
        code2id = {c: i for i, c in enumerate(node_df["code"])}
        edge_elasticity = []
        
        for _, row in edge_df.iterrows():
            if row["source"] in code2id and row["target"] in code2id:
                src_id = code2id[row["source"]]
                tgt_id = code2id[row["target"]]
                # 공급자와 수요자의 평균 탄성
                avg_elasticity = (elasticity_scores[src_id] + elasticity_scores[tgt_id]) / 2
                edge_elasticity.append(avg_elasticity)
        
        return torch.tensor(edge_elasticity, dtype=torch.float)
    
    def create_esg_channels(self, node_df: pd.DataFrame, feat_cols: List[str]) -> Dict[str, torch.Tensor]:
        """E/S/G 채널별 특성 생성"""
        channels = {}
        node_features = node_df[feat_cols].values
        
        # E(Environmental): 가격지수 변동성 (PII, PGO)
        if "PII" in feat_cols and "PGO" in feat_cols:
            pii_idx = feat_cols.index("PII")
            pgo_idx = feat_cols.index("PGO")
            env_volatility = np.abs(node_features[:, pii_idx]) + np.abs(node_features[:, pgo_idx])
            channels["E"] = torch.tensor(env_volatility, dtype=torch.float)
        else:
            channels["E"] = torch.ones(len(node_df), dtype=torch.float)
        
        # S(Social): 노동 집약도 (RVA/GO)
        if "RVA" in feat_cols and "GO" in feat_cols:
            rva_idx = feat_cols.index("RVA")
            go_idx = feat_cols.index("GO")
            labor_intensity = node_features[:, rva_idx] / (node_features[:, go_idx] + 1e-8)
            channels["S"] = torch.tensor(labor_intensity, dtype=torch.float)
        else:
            channels["S"] = torch.ones(len(node_df), dtype=torch.float)
        
        # G(Governance): 산업 코드 기반 (금융, 부동산 등)
        governance_scores = []
        for code in node_df["code"]:
            if code.startswith("K") or code.startswith("L"):  # 금융, 부동산
                governance_scores.append(1.0)
            elif code.startswith("O"):  # 정부
                governance_scores.append(0.8)
            else:
                governance_scores.append(0.5)
        
        channels["G"] = torch.tensor(governance_scores, dtype=torch.float)
        
        print(f"✅ ESG 채널: E={channels['E'].shape}, S={channels['S'].shape}, G={channels['G'].shape}")
        return channels
    
    def compute_leontief_prior(self, edge_df: pd.DataFrame, node_df: pd.DataFrame) -> torch.Tensor:
        """Leontief 역행렬 계산"""
        codes = node_df["code"].tolist()
        n = len(codes)
        code2id = {c: i for i, c in enumerate(codes)}
        
        # 기술계수 행렬 A
        A = torch.zeros((n, n), dtype=torch.float)
        total_inputs = torch.zeros(n, dtype=torch.float)
        
        # 총 투입 계산
        for _, row in edge_df.iterrows():
            if row["target"] in code2id:
                tgt_id = code2id[row["target"]]
                total_inputs[tgt_id] += row["weight"]
        
        # 기술계수 계산
        for _, row in edge_df.iterrows():
            if row["source"] in code2id and row["target"] in code2id:
                src_id = code2id[row["source"]]
                tgt_id = code2id[row["target"]]
                if total_inputs[tgt_id] > 0:
                    A[src_id, tgt_id] = row["weight"] / total_inputs[tgt_id]
        
        # Leontief 역행렬: (I - A)^(-1)
        I = torch.eye(n, dtype=torch.float)
        try:
            leontief_inverse = torch.inverse(I - A)
        except:
            leontief_inverse = torch.pinverse(I - A)
        
        print(f"✅ Leontief 역행렬: {leontief_inverse.shape}")
        return leontief_inverse
    
    def create_temporal_tensor(self, temporal_data: Dict[str, pd.DataFrame], 
                              node_df: pd.DataFrame, feat_cols: List[str]) -> torch.Tensor:
        """시차 특성 텐서 생성"""
        if not temporal_data:
            # 기본값: 현재 특성을 복제하여 시차 생성
            x_current = node_df[feat_cols].values
            x_temporal = np.tile(x_current[:, np.newaxis, :], (1, self.lag_periods, 1))
            return torch.tensor(x_temporal, dtype=torch.float)
        
        codes = node_df["code"].tolist()
        n_nodes = len(codes)
        n_features = len(feat_cols)
        
        # 시차별 특성 행렬 초기화
        x_temporal = np.zeros((n_nodes, self.lag_periods, n_features))
        
        # 각 지표별 시차 데이터 채우기
        for feat_idx, feat in enumerate(feat_cols):
            if feat in temporal_data:
                temp_df = temporal_data[feat]
                code2id = {c: i for i, c in enumerate(codes)}
                
                # 시간 컬럼들 (feat_ 접두사 제거한 컬럼들)
                time_cols = [c for c in temp_df.columns if c.startswith(feat + "_")]
                time_cols = sorted(time_cols)[-self.lag_periods:]  # 최신 lag_periods 개
                
                for code, row in temp_df.iterrows():
                    if temp_df.loc[code, "code"] in code2id:
                        node_idx = code2id[temp_df.loc[code, "code"]]
                        for t, col in enumerate(time_cols):
                            if t < self.lag_periods and col in temp_df.columns:
                                x_temporal[node_idx, t, feat_idx] = temp_df.loc[code, col]
        
        print(f"✅ 시차 텐서: {x_temporal.shape}")
        return torch.tensor(x_temporal, dtype=torch.float)


# ---------------------------------------------------------------------
# 고급 GNN 모델들 (이전과 동일)
# ---------------------------------------------------------------------

class BiDirectionalGNN(nn.Module):
    """이중 방향 메시지 패싱 GNN"""
    
    def __init__(self, input_dim: int, hidden_dim: int, num_heads: int = 4):
        super().__init__()
        self.upstream_gnn = GATConv(input_dim, hidden_dim, heads=num_heads, concat=True)
        self.downstream_gnn = GATConv(input_dim, hidden_dim, heads=num_heads, concat=True)
        self.fusion = nn.Linear(hidden_dim * num_heads * 2, hidden_dim)
        
    def forward(self, x: torch.Tensor, edge_index_up: torch.Tensor, 
                edge_index_down: torch.Tensor, edge_attr_up: torch.Tensor = None,
                edge_attr_down: torch.Tensor = None) -> torch.Tensor:
        h_up = self.upstream_gnn(x, edge_index_up, edge_attr_up)
        h_down = self.downstream_gnn(x, edge_index_down, edge_attr_down)
        h_combined = torch.cat([h_up, h_down], dim=1)
        h_fused = F.relu(self.fusion(h_combined))
        return h_fused, h_up, h_down


class ESGChannelGating(nn.Module):
    """E/S/G 채널별 게이팅"""
    
    def __init__(self, hidden_dim: int):
        super().__init__()
        self.gate_E = nn.Linear(hidden_dim, hidden_dim)
        self.gate_S = nn.Linear(hidden_dim, hidden_dim)
        self.gate_G = nn.Linear(hidden_dim, hidden_dim)
        
    def forward(self, h: torch.Tensor, esg_channels: Dict[str, torch.Tensor]) -> torch.Tensor:
        batch_size = h.size(0)
        gates = []
        
        for channel in ["E", "S", "G"]:
            if channel in esg_channels and len(esg_channels[channel]) == batch_size:
                channel_input = esg_channels[channel].unsqueeze(1).expand_as(h)
                gate = torch.sigmoid(getattr(self, f"gate_{channel}")(channel_input))
                gates.append(gate)
            else:
                gates.append(torch.ones_like(h))
        
        gated_h = h * (gates[0] + gates[1] + gates[2]) / 3
        return gated_h


class IntegratedESGGNN(nn.Module):
    """통합 ESG-GNN 모델"""
    
    def __init__(self, input_dim: int, hidden_dim: int, num_nodes: int, 
                 lag_periods: int = 4, num_heads: int = 4):
        super().__init__()
        
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.num_nodes = num_nodes
        
        # 입력 임베딩
        self.input_embedding = nn.Linear(input_dim, hidden_dim)
        
        # 핵심 GNN 레이어들
        self.bidirectional_gnn = BiDirectionalGNN(hidden_dim, hidden_dim, num_heads)
        self.esg_gating = ESGChannelGating(hidden_dim)
        
        # Leontief 융합
        self.adj_residual = nn.Parameter(torch.randn(num_nodes, num_nodes) * 0.01)
        self.fusion_weight = nn.Parameter(torch.tensor(0.5))
        
        # 시간 처리
        self.temporal_lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        
        # 멀티태스크 출력
        self.rva_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        self.price_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        self.intermediate_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        # Attention for interpretability
        self.attention = nn.MultiheadAttention(hidden_dim, num_heads)
        
    def forward(self, x: torch.Tensor, edge_index_up: torch.Tensor,
                edge_index_down: torch.Tensor, edge_attr_up: torch.Tensor,
                edge_attr_down: torch.Tensor, esg_channels: Dict[str, torch.Tensor],
                leontief_prior: torch.Tensor, x_temporal: torch.Tensor = None) -> Dict[str, torch.Tensor]:
        
        # 입력 임베딩
        h = F.relu(self.input_embedding(x))
        
        # 이중 방향 GNN
        h_bidirectional, h_up, h_down = self.bidirectional_gnn(
            h, edge_index_up, edge_index_down, edge_attr_up, edge_attr_down
        )
        
        # ESG 채널 게이팅
        h_gated = self.esg_gating(h_bidirectional, esg_channels)
        
        # Leontief 융합
        adj_learned = leontief_prior + self.adj_residual
        adj_normalized = F.softmax(adj_learned, dim=1)
        h_leontief = torch.mm(adj_normalized, h_gated)
        h_fused = self.fusion_weight * h_leontief + (1 - self.fusion_weight) * h_gated
        
        # 시간 차원 처리 (선택적)
        if x_temporal is not None and x_temporal.size(1) > 1:
            # 시간 임베딩
            batch_size, seq_len, feat_dim = x_temporal.shape
            x_temp_flat = x_temporal.view(-1, feat_dim)
            h_temp_flat = F.relu(self.input_embedding(x_temp_flat))
            h_temp_seq = h_temp_flat.view(batch_size, seq_len, -1)
            
            # LSTM으로 시간 의존성 학습
            h_temporal_out, _ = self.temporal_lstm(h_temp_seq)
            h_temporal_final = h_temporal_out[:, -1, :]  # 마지막 시점
            
            # 현재와 시간 정보 결합
            h_final = (h_fused + h_temporal_final) / 2
        else:
            h_final = h_fused
        
        # Attention for interpretability
        h_attended, attention_weights = self.attention(
            h_final.unsqueeze(0), h_final.unsqueeze(0), h_final.unsqueeze(0)
        )
        h_final = h_attended.squeeze(0)
        
        # 멀티태스크 예측
        predictions = {
            'rva_change': self.rva_head(h_final).squeeze(-1),
            'price_change': self.price_head(h_final).squeeze(-1),
            'intermediate_change': self.intermediate_head(h_final).squeeze(-1),
            'embeddings': h_final,
            'attention_weights': attention_weights,
            'adj_learned': adj_learned,
            'upstream_embeddings': h_up,
            'downstream_embeddings': h_down
        }
        
        return predictions


# ---------------------------------------------------------------------
# 통합 실행 함수
# ---------------------------------------------------------------------

def run_integrated_esg_pipeline():
    """실제 데이터를 사용한 통합 ESG-GNN 파이프라인"""
    
    print("🚀 통합 ESG-GNN 파이프라인 시작...")
    
    # 파일 존재 확인
    missing_files = []
    if not os.path.exists(IOT_PATH):
        missing_files.append(f"IOT: {IOT_PATH}")
    
    for name, path in FILES6.items():
        if not os.path.exists(path):
            missing_files.append(f"{name}: {path}")
    
    if missing_files:
        print("❌ 누락된 파일들:")
        for file in missing_files:
            print(f"   - {file}")
        print("\n📝 데모 모드로 실행됩니다...")
        return run_demo_mode()
    
    try:
        # 1. 데이터 전처리
        processor = IntegratedESGProcessor(IOT_PATH, FILES6, NAME2CODE, lag_periods=4)
        data = processor.process_full_pipeline()
        
        print(f"✅ 데이터 전처리 완료:")
        print(f"   - 노드 수: {data['x'].shape[0]}")
        print(f"   - 특성 수: {data['x'].shape[1]}")
        print(f"   - 엣지 수: {data['edge_index_up'].shape[1]}")
        print(f"   - 산업 코드: {len(data['codes'])}개")
        
        # 2. 모델 초기화
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"📱 디바이스: {device}")
        
        model = IntegratedESGGNN(
            input_dim=data['x'].shape[1],
            hidden_dim=64,
            num_nodes=data['x'].shape[0],
            lag_periods=4,
            num_heads=4
        ).to(device)
        
        # 3. 데이터를 디바이스로 이동
        for key in ['x', 'edge_index_up', 'edge_index_down', 'edge_weight_up', 
                   'edge_weight_down', 'leontief_prior', 'x_temporal']:
            if key in data:
                data[key] = data[key].to(device)
        
        for channel in data['esg_channels']:
            data['esg_channels'][channel] = data['esg_channels'][channel].to(device)
        
        # 4. 모델 테스트
        model.eval()
        with torch.no_grad():
            predictions = model(
                data['x'],
                data['edge_index_up'], 
                data['edge_index_down'],
                data['edge_weight_up'],
                data['edge_weight_down'],
                data['esg_channels'],
                data['leontief_prior'],
                data['x_temporal']
            )
        
        print(f"✅ 모델 예측 완료:")
        print(f"   - RVA 변화: {predictions['rva_change'].shape}")
        print(f"   - 가격 변화: {predictions['price_change'].shape}")
        print(f"   - 중간투입 변화: {predictions['intermediate_change'].shape}")
        
        # 5. 주요 분석 결과
        analyze_results(data, predictions)
        
        return {
            'data': data,
            'model': model,
            'predictions': predictions,
            'device': device
        }
        
    except Exception as e:
        print(f"❌ 실제 데이터 처리 실패: {e}")
        print("📝 데모 모드로 전환...")
        return run_demo_mode()


def run_demo_mode():
    """데모 모드 실행 (실제 파일이 없을 때)"""
    print("🎮 데모 모드 실행...")
    
    # 가상 데이터 생성
    num_nodes = 50
    num_features = 6
    num_edges = 200
    
    # 기본 특성
    x = torch.randn(num_nodes, num_features)
    
    # 엣지 인덱스
    edge_index = torch.randint(0, num_nodes, (2, num_edges))
    edge_weight = torch.rand(num_edges) + 0.1
    
    # ESG 채널
    esg_channels = {
        'E': torch.rand(num_nodes),
        'S': torch.rand(num_nodes),
        'G': torch.rand(num_nodes)
    }
    
    # Leontief prior
    leontief_prior = torch.eye(num_nodes) * 0.5 + torch.rand(num_nodes, num_nodes) * 0.1
    
    # 시차 데이터
    x_temporal = torch.randn(num_nodes, 4, num_features)
    
    # 가상 코드
    codes = [f"DEMO_{i:03d}" for i in range(num_nodes)]
    
    data = {
        'x': x,
        'edge_index_up': edge_index,
        'edge_index_down': edge_index,
        'edge_weight_up': edge_weight,
        'edge_weight_down': edge_weight,
        'esg_channels': esg_channels,
        'leontief_prior': leontief_prior,
        'x_temporal': x_temporal,
        'codes': codes
    }
    
    # 모델 초기화
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = IntegratedESGGNN(
        input_dim=num_features,
        hidden_dim=32,
        num_nodes=num_nodes,
        lag_periods=4,
        num_heads=2
    ).to(device)
    
    # 데이터를 디바이스로 이동
    for key in ['x', 'edge_index_up', 'edge_index_down', 'edge_weight_up', 
               'edge_weight_down', 'leontief_prior', 'x_temporal']:
        data[key] = data[key].to(device)
    
    for channel in data['esg_channels']:
        data['esg_channels'][channel] = data['esg_channels'][channel].to(device)
    
    # 예측
    model.eval()
    with torch.no_grad():
        predictions = model(
            data['x'],
            data['edge_index_up'],
            data['edge_index_down'], 
            data['edge_weight_up'],
            data['edge_weight_down'],
            data['esg_channels'],
            data['leontief_prior'],
            data['x_temporal']
        )
    
    print(f"✅ 데모 예측 완료:")
    print(f"   - 노드 수: {num_nodes}")
    print(f"   - 엣지 수: {num_edges}")
    print(f"   - 예측 차원: {predictions['rva_change'].shape}")
    
    # 데모 분석
    analyze_demo_results(data, predictions)
    
    return {
        'data': data,
        'model': model,
        'predictions': predictions,
        'device': device,
        'mode': 'demo'
    }


def analyze_results(data, predictions):
    """실제 데이터 분석 결과"""
    print("\n📊 === 실제 데이터 분석 결과 ===")
    
    # 상위 위험 산업 식별
    rva_risk = torch.abs(predictions['rva_change'])
    price_risk = torch.abs(predictions['price_change'])
    
    combined_risk = 0.6 * rva_risk + 0.4 * price_risk
    top_risk_indices = torch.topk(combined_risk, k=min(10, len(combined_risk))).indices
    
    print("🔥 상위 위험 산업:")
    for i, idx in enumerate(top_risk_indices):
        code = data['codes'][idx]
        risk_score = combined_risk[idx].item()
        rva_change = predictions['rva_change'][idx].item()
        price_change = predictions['price_change'][idx].item()
        print(f"   {i+1}. {code}: 위험도={risk_score:.3f} (RVA={rva_change:.3f}, Price={price_change:.3f})")
    
    # ESG 채널별 영향도
    print("\n🌍 ESG 채널별 평균 영향도:")
    for channel, values in data['esg_channels'].items():
        avg_impact = values.mean().item()
        std_impact = values.std().item()
        print(f"   {channel}: {avg_impact:.3f} ± {std_impact:.3f}")
    
    # 네트워크 특성
    num_nodes = data['x'].shape[0]
    num_edges = data['edge_index_up'].shape[1]
    density = num_edges / (num_nodes * (num_nodes - 1))
    
    print(f"\n🕸️ 네트워크 특성:")
    print(f"   밀도: {density:.4f}")
    print(f"   평균 연결도: {num_edges / num_nodes:.1f}")
    
    # 예측 분포
    print(f"\n📈 예측 분포:")
    for task in ['rva_change', 'price_change', 'intermediate_change']:
        values = predictions[task]
        print(f"   {task}: 평균={values.mean().item():.4f}, 표준편차={values.std().item():.4f}")


def analyze_demo_results(data, predictions):
    """데모 데이터 분석 결과"""
    print("\n🎮 === 데모 분석 결과 ===")
    
    # 기본 통계
    print("📊 예측 통계:")
    for task in ['rva_change', 'price_change', 'intermediate_change']:
        values = predictions[task]
        print(f"   {task}: 범위=[{values.min().item():.3f}, {values.max().item():.3f}]")
    
    # 가상 시나리오
    print("\n🎯 가상 ESG 충격 시나리오:")
    
    # 시나리오 1: 탄소세 도입
    carbon_sensitive = data['esg_channels']['E'] > data['esg_channels']['E'].median()
    carbon_impact = predictions['price_change'][carbon_sensitive].mean()
    print(f"   탄소세 영향 (E채널 높은 산업): {carbon_impact.item():.3f}")
    
    # 시나리오 2: 노동규제 강화  
    labor_sensitive = data['esg_channels']['S'] > data['esg_channels']['S'].median()
    labor_impact = predictions['rva_change'][labor_sensitive].mean()
    print(f"   노동규제 영향 (S채널 높은 산업): {labor_impact.item():.3f}")
    
    # 시나리오 3: 금융규제
    finance_sensitive = data['esg_channels']['G'] > data['esg_channels']['G'].median()
    finance_impact = predictions['intermediate_change'][finance_sensitive].mean()
    print(f"   금융규제 영향 (G채널 높은 산업): {finance_impact.item():.3f}")


def create_esg_shock_scenario(data, model, shock_type: str = "carbon_tax", 
                            shock_intensity: float = 0.1):
    """ESG 충격 시나리오 시뮬레이션"""
    print(f"\n⚡ {shock_type.upper()} 충격 시나리오 (강도: {shock_intensity})")
    
    # 원본 데이터 복사
    x_shocked = data['x'].clone()
    
    if shock_type == "carbon_tax":
        # 에너지/화학 산업에 가격 충격
        energy_codes = [i for i, code in enumerate(data['codes']) 
                       if any(prefix in code for prefix in ['B05', 'C19', 'D'])]
        if energy_codes:
            x_shocked[energy_codes, 2] += shock_intensity  # PII 지수 상승
            print(f"   영향 받은 산업: {len(energy_codes)}개")
    
    elif shock_type == "labor_regulation":
        # 노동집약 산업에 비용 충격
        labor_intensive = data['esg_channels']['S'] > data['esg_channels']['S'].quantile(0.7)
        x_shocked[labor_intensive, 0] -= shock_intensity  # RVA 감소
        print(f"   영향 받은 산업: {labor_intensive.sum()}개")
    
    elif shock_type == "financial_regulation":
        # 금융/부동산에 중간투입 충격
        finance_codes = [i for i, code in enumerate(data['codes']) 
                        if any(prefix in code for prefix in ['K', 'L'])]
        if finance_codes:
            x_shocked[finance_codes, 1] += shock_intensity  # RII 증가
            print(f"   영향 받은 산업: {len(finance_codes)}개")
    
    # 충격 후 예측
    model.eval()
    with torch.no_grad():
        predictions_shocked = model(
            x_shocked,
            data['edge_index_up'],
            data['edge_index_down'],
            data['edge_weight_up'],
            data['edge_weight_down'],
            data['esg_channels'],
            data['leontief_prior'],
            data['x_temporal']
        )
    
    # 원본과 비교
    original_predictions = model(
        data['x'],
        data['edge_index_up'],
        data['edge_index_down'],
        data['edge_weight_up'],
        data['edge_weight_down'],
        data['esg_channels'],
        data['leontief_prior'],
        data['x_temporal']
    )
    
    # 변화량 계산
    rva_delta = predictions_shocked['rva_change'] - original_predictions['rva_change']
    price_delta = predictions_shocked['price_change'] - original_predictions['price_change']
    
    print(f"   RVA 평균 변화: {rva_delta.mean().item():.4f}")
    print(f"   가격 평균 변화: {price_delta.mean().item():.4f}")
    
    # 상위 영향 받은 산업
    total_impact = torch.abs(rva_delta) + torch.abs(price_delta)
    top_impacted = torch.topk(total_impact, k=5).indices
    
    print("   상위 영향 산업:")
    for i, idx in enumerate(top_impacted):
        code = data['codes'][idx]
        impact = total_impact[idx].item()
        print(f"      {i+1}. {code}: {impact:.4f}")
    
    return {
        'rva_delta': rva_delta,
        'price_delta': price_delta,
        'total_impact': total_impact,
        'top_impacted_codes': [data['codes'][i] for i in top_impacted]
    }


def demonstrate_advanced_features():
    """고급 기능 시연"""
    print("\n🔬 === 고급 기능 시연 ===")
    
    # 파이프라인 실행
    results = run_integrated_esg_pipeline()
    
    if results is None:
        return
    
    data = results['data']
    model = results['model']
    
    # 1. ESG 충격 시나리오들
    scenarios = ['carbon_tax', 'labor_regulation', 'financial_regulation']
    intensities = [0.05, 0.1, 0.15]
    
    print("\n⚡ 다양한 ESG 충격 시나리오:")
    scenario_results = {}
    
    for scenario in scenarios:
        for intensity in intensities:
            result = create_esg_shock_scenario(data, model, scenario, intensity)
            scenario_results[f"{scenario}_{intensity}"] = result
    
    # 2. 시나리오 간 비교
    print("\n📊 시나리오 영향 비교:")
    for key, result in scenario_results.items():
        avg_impact = result['total_impact'].mean().item()
        max_impact = result['total_impact'].max().item()
        print(f"   {key}: 평균={avg_impact:.4f}, 최대={max_impact:.4f}")
    
    # 3. 정책 권고사항
    print("\n📋 정책 권고사항:")
    print("   1. 탄소세: 에너지 전환 지원 정책 병행 필요")
    print("   2. 노동규제: 단계적 도입으로 충격 완화")
    print("   3. 금융규제: 중소기업 유동성 지원 강화")
    
    return scenario_results


# 메인 실행부
if __name__ == "__main__":
    print("🌟 === 통합 ESG-GNN 분석 시스템 ===")
    print("이 시스템은 다음 기능을 제공합니다:")
    print("📈 실시간 ESG 리스크 모니터링")
    print("🔍 충격 전파 경로 분석")
    print("⚡ 정책 시나리오 시뮬레이션")
    print("🎯 멀티태스크 경제 지표 예측")
    print("🤖 Graph-AI 기반 인사이트 생성")
    print()
    
    # 고급 기능 시연
    results = demonstrate_advanced_features()
    
    print("\n✨ === 분석 완료 ===")
    print("💡 주요 특징:")
    print("   - 이중 방향 메시지 패싱으로 공급-수요 관계 모델링")
    print("   - E/S/G 채널별 차별화된 충격 전파")
    print("   - Leontief 경제학 이론과 GNN 결합")
    print("   - 시차 효과를 고려한 동적 분석")
    print("   - 실시간 이상치 탐지 및 조기 경보")
    print("   - 해석 가능한 경로 기여도 분석")
    
    print("\n🚀 다음 단계 개발 권장사항:")
    print("   1. 실제 BEA 데이터 API 연동")
    print("   2. 웹 대시보드 개발 (Streamlit/Dash)")  
    print("   3. 실시간 뉴스/정책 연동")
    print("   4. MLOps 파이프라인 구축")
    print("   5. 규제기관 리포트 자동 생성") 
    "